## Caution
Download finetuning_utils from https://github.com/skrhakv/cryptic-finetuning/blob/master/src/finetuning_utils.py

In [ ]:
import torch
from transformers import AutoTokenizer
import finetuning_utils # download from https://github.com/skrhakv/cryptic-finetuning/blob/master/src/finetuning_utils.py

MAX_LENGTH = 1024
MODEL_NAME = 'facebook/esm2_t36_3B_UR50D'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

MODEL_PATH = "/path/to/cbs-model.pt" # download from https://cunicz-my.sharepoint.com/:u:/g/personal/99575159_cuni_cz/IQDTozFMT5pMQoWIaHzqQmQ5ASbN36kTO4OUv2GFxuu2pno?e=iOakb4
model = torch.load(MODEL_PATH, weights_only=False)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


In [ ]:
model.eval()

KRAS_sequence = 'GMTEYKLVVVGACGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVIDGETSLLDILDTAGQEEYSAMRDQYMRTGEGFLLVFAINNTKSFEDIHHYREQIKRVKDSEDVPMVLVGNKSDLPSRTVDTKQAQDLARSYGIPFIETSAKTRQGVDDAFYTLVREIRKHKEK'

# tokenize the sequence
tokenized_sequences = tokenizer(KRAS_sequence, max_length=MAX_LENGTH, padding='max_length', truncation=True)
tokenized_sequences = {k: torch.tensor([v]).to(device) for k,v in tokenized_sequences.items()}

# predict
output, _, _ = model(tokenized_sequences)
output = output.flatten()

mask = (tokenized_sequences['attention_mask'] == 1).flatten()

output = torch.sigmoid(output[mask][1:-1]).detach().cpu().numpy()
print(output) # returns probabilities for each amino acid in the sequence being a cryptic site
assert len(output) == len(KRAS_sequence) # should be the same length as the input sequence

[0.0538  0.1373  0.2605  0.3574  0.3616  0.4905  0.2947  0.63    0.1489
 0.5815  0.7446  0.933   0.9717  0.9526  0.949   0.912   0.9556  0.942
 0.898   0.624   0.1903  0.3728  0.283   0.2185  0.1729  0.211   0.1837
 0.3337  0.885   0.897   0.8438  0.897   0.9014  0.856   0.8657  0.8496
 0.68    0.7373  0.5864  0.4846  0.1853  0.412   0.1417  0.157   0.1409
 0.09283 0.1305  0.10876 0.0802  0.11694 0.05954 0.075   0.11816 0.07697
 0.2637  0.2108  0.5923  0.857   0.8823  0.861   0.829   0.844   0.7236
 0.6777  0.5615  0.3718  0.556   0.749   0.7344  0.4424  0.6353  0.8296
 0.6855  0.713   0.6245  0.52    0.4878  0.09705 0.6167  0.058   0.0922
 0.0763  0.0725  0.2583  0.1716  0.1392  0.0904  0.0629  0.2084  0.1184
 0.06696 0.0802  0.2612  0.06915 0.10144 0.3843  0.5386  0.1844  0.1633
 0.5923  0.456   0.1896  0.5435  0.7666  0.789   0.7686  0.5376  0.3826
 0.2502  0.4578  0.1487  0.1501  0.0542  0.0666  0.0695  0.0885  0.9165
 0.958   0.768   0.9536  0.891   0.3074  0.138   0.1605  0.1415 